In [84]:
import torch
import pandas as pd
import nltk

print("PyTorch version:", torch.__version__)
print("Pandas loaded successfully")
print("NLTK loaded successfully")

PyTorch version: 2.12.0+cpu
Pandas loaded successfully
NLTK loaded successfully


In [85]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.12.0+cpu
False


In [87]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Hp\Desktop\Sentiment-Analysis-Project\data\IMDB Dataset.csv")

print(df.head())
print(df.info())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB
None
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [88]:
df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

print(df.head())

                                              review sentiment  label
0  One of the other reviewers has mentioned that ...  positive      1
1  A wonderful little production. <br /><br />The...  positive      1
2  I thought this was a wonderful way to spend ti...  positive      1
3  Basically there's a family where a little boy ...  negative      0
4  Petter Mattei's "Love in the Time of Money" is...  positive      1


In [89]:
print(df.isnull().sum())

review       0
sentiment    0
label        0
dtype: int64


Train-Test Split

In [90]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(40000,) (10000,)


In [92]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'<.*?>', '', text)  # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

In [93]:
X_train = X_train.apply(clean_text)
X_test = X_test.apply(clean_text)

print(X_train.iloc[0])

i caught this little gem totally by accident back in or i was at a revival theatre to see two old silly scifi movies the theatre was packed full and with no warning they showed a bunch of scifi short spoofs to get us in the mood most were somewhat amusing but this came on and within seconds the audience was in hysterics the biggest laugh came when they showed princess laia having huge cinnamon buns instead of hair on her head she looks at the camera gives a grim smile and nods that made it even funnier you gotta see chewabacca played by what looks like a muppet it was extremely silly and stupidbut i couldnt stop laughing most of the dialogue was drowned out because of all the laughter also if you know star wars pretty well its even funnierthey deliberately poke fun at some of the dialogue this really works with an audience a definite


In [94]:
def tokenize(text):
    return text.split()

In [95]:
print(tokenize("this movie is great"))

['this', 'movie', 'is', 'great']


In [96]:
from collections import Counter

def build_vocab(texts, max_size=10000):
    counter = Counter()

    for text in texts:
        tokens = tokenize(text)
        counter.update(tokens)

    # Most common words
    vocab = counter.most_common(max_size)

    # Create word → index mapping
    word2idx = {'<PAD>': 0, '<UNK>': 1}

    for i, (word, _) in enumerate(vocab, start=2):
        word2idx[word] = i

    return word2idx

In [97]:
word2idx = build_vocab(X_train)

print("Vocab size:", len(word2idx))

Vocab size: 10002


In [98]:
def encode(text, word2idx):
    tokens = tokenize(text)
    return [
        word2idx.get(token, word2idx['<UNK>'])
        for token in tokens
    ]

In [100]:
sample = X_train.iloc[0]
print(sample)

encoded = encode(sample, word2idx)
print(encoded[:20])

i caught this little gem totally by accident back in or i was at a revival theatre to see two old silly scifi movies the theatre was packed full and with no warning they showed a bunch of scifi short spoofs to get us in the mood most were somewhat amusing but this came on and within seconds the audience was in hysterics the biggest laugh came when they showed princess laia having huge cinnamon buns instead of hair on her head she looks at the camera gives a grim smile and nods that made it even funnier you gotta see chewabacca played by what looks like a muppet it was extremely silly and stupidbut i couldnt stop laughing most of the dialogue was drowned out because of all the laughter also if you know star wars pretty well its even funnierthey deliberately poke fun at some of the dialogue this really works with an audience a definite
[10, 1014, 11, 116, 1463, 438, 32, 1599, 142, 8, 39, 10, 13, 30, 4, 8483, 1618, 6, 64, 107]


In [101]:
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))  # 0 = <PAD>
    else:
        return seq[:max_len]

In [102]:
MAX_LEN = 200

In [103]:
import sys
import os

# go one level up (from notebooks → project root)
project_root = os.path.abspath("..")

# now go into src
src_path = os.path.join(project_root, "src")

sys.path.append(src_path)

print(src_path)
print(os.listdir(src_path))  # should show dataset.py

c:\Users\Hp\Desktop\Sentiment-Analysis-Project\src
['dataset.py', 'model.py', 'model_v2.py', 'train.py', 'utils.py', '__init__.py', '__pycache__']


In [105]:
from dataset import IMDBDataset

train_dataset = IMDBDataset(X_train, y_train, word2idx, MAX_LEN)
test_dataset = IMDBDataset(X_test, y_test, word2idx, MAX_LEN)

print(len(train_dataset))

40000


In [106]:
sample_x, sample_y = train_dataset[0]
print(sample_x.shape)
print(sample_y) 

torch.Size([200])
tensor(1)


In [107]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [108]:
for batch_x, batch_y in train_loader:
    print(batch_x.shape)
    print(batch_y.shape)
    break

torch.Size([32, 200])
torch.Size([32])


In [109]:
from model_v2 import SentimentModel
vocab_size = len(word2idx)
embed_dim = 128
hidden_dim = 128

model = SentimentModel(vocab_size, embed_dim, hidden_dim)

print(model)

SentimentModel(
  (embedding): Embedding(10002, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [110]:
for batch_x, batch_y in train_loader:
    output = model(batch_x)
    print(output.shape)
    break

torch.Size([32, 1])


In [111]:
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

In [112]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [113]:
batch_y = batch_y.float().unsqueeze(1)

In [115]:
for batch_x, batch_y in train_loader:
    batch_y = batch_y.float().unsqueeze(1)

    # forward pass
    outputs = model(batch_x)

    # loss
    loss = criterion(outputs, batch_y)

    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Loss:", loss.item())
    break

Loss: 0.7138121724128723


In [127]:
EPOCHS = 5

for epoch in range(EPOCHS):
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_y = batch_y.float().unsqueeze(1)

        # forward
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 0.3657
Epoch 2, Loss: 0.2793
Epoch 3, Loss: 0.2342
Epoch 4, Loss: 0.1989
Epoch 5, Loss: 0.1758


In [128]:
model.eval()

SentimentModel(
  (embedding): Embedding(10002, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_y = batch_y.float().unsqueeze(1)
        outputs = model(batch_x)
        # simpler correct version
        preds = (outputs >= 0).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
accuracy = correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 0.8658


In [ ]:
#TESTING

In [ ]:
def predict(text, model, word2idx, max_len):
    model.eval()
    # clean text (reuse your function)
    text = clean_text(text)
    # tokenize
    tokens = text.split()
    # encode
    encoded = [
        word2idx.get(token, word2idx['<UNK>'])
        for token in tokens
    ]
    # pad
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    # convert to tensor
    input_tensor = torch.tensor(encoded).unsqueeze(0)
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.sigmoid(output)
        prediction = (prob >= 0.5).float().item()
    return "Positive" if prediction == 1 else "Negative"

In [ ]:
print(predict("This movie was absolutely amazing, I loved it!", model, word2idx, MAX_LEN))
print(predict("This was the worst film I have ever seen.", model, word2idx, MAX_LEN))

Positive
Negative


In [121]:
print(predict("Amazing movie!", model, word2idx, MAX_LEN))
print(predict("Terrible film.", model, word2idx, MAX_LEN))

Positive
Negative


In [ ]:
print(predict(
    "This movie started off really slow and I almost stopped watching, but the second half was absolutely brilliant with great acting and an emotional storyline that really stayed with me.",
    model, word2idx, MAX_LEN
))
print(predict(
    "I had high expectations from this movie but it completely disappointed me with poor acting, weak storyline, and unnecessary scenes that made it boring and frustrating to watch.",
    model, word2idx, MAX_LEN
))

Positive
Negative


In [ ]:
print(predict(
    "The movie had great visuals but the story was boring.",
    model, word2idx, MAX_LEN
))
print(predict(
    "Not bad, but not something I would watch again.",
    model, word2idx, MAX_LEN
))
print(predict(
    "The acting was good, but the plot was terrible and confusing.",
    model, word2idx, MAX_LEN
))

Positive
Negative
Negative


In [124]:
print(predict(
    "I absolutely loved this movie, the acting was fantastic, the story was engaging, and it was one of the best films I have seen in years",
    model, word2idx, MAX_LEN
))

Positive


In [130]:
torch.save(model.state_dict(), "sentiment_model.pth")

In [131]:
torch.save(word2idx, "word2idx.pt")